# Decision Analysis Data Preparation

This notebook prepares land-building, Habitat Suitability Index (HSI), project cost, and fresh-marsh metrics for decision analysis.

Final dataset level:

`project_id × ScenarioID × Region × implementation_period`

*Resources:*

[Master Plan API Documentation](https://github.com/pscedu/cpra.mp.data/tree/main)

## Load and Prepare Data
Imports, parameters, database connection, and project/ecoregion scope setup.

In [1]:
import getpass
import itertools
import time
from collections import defaultdict
from IPython.display import display
from contextlib import ExitStack

import altair as alt
import numpy as np
import pandas as pd
import polars as pl
import rasterio as rio
from polars import col
from sqlalchemy import create_engine

from cpra.mp.data import read_data

In [2]:
t0 = time.time()

# Change paramters as desired 
scenario_list = [7, 8]

model_group_id_list = [602, 608, 520]
selected_unit = 'sq_meters' # Other option: 'sq_meters'

# Future Without Action (FWOA) 
fwoa_model_group = 520
 
# Master Plan Attribute Database (MAD) Table Names - Tables that are defining how projects are linked to model groups and ecoregions
A_ECOREGION = "gsd.a_ecoregion"
PROJECT_ECOREGION_CROSSWALK = "cma.c_project_ecoregion"
PROJECT_MODEL_GROUP_CROSSWALK = "cma.c_project_model_group"
A_PROJECT = 'cma.a_project'

# Link to Morph pixel to Ecoregion ID Crosswalk
MORPH_TO_ECOREGION_CROSSWALK = r"/ocean/projects/bcs200002p/shared/grids/crosswalks/morph_pixel_v001__ecoregion_v001.tif"

# Switch for units 
valid_units = {"acres", "sq_meters"} 
conversion_factor = 4046.86 if selected_unit == "acres" else 1.0

In [5]:
# MAD Connection Properties
username = getpass.getuser()
password = getpass.getpass("Enter your password: ")
port = "5432"
host = "vm002.bridges2.psc.edu"
db_name = "mpd_dev"
uri = f"postgresql://{username}:{password}@{host}:{port}/{db_name}"
engine = create_engine(uri)

# NOTE - this will be incorporated into cpra.mp.data library


In [6]:
# Load MAD Tables

# Load a_ecoregion
query = f"SELECT * FROM {A_ECOREGION}"
a_ecoregion = pl.from_pandas(pd.read_sql(query, engine))

# Load c_project_ecoregion
query = f"SELECT * FROM {PROJECT_ECOREGION_CROSSWALK} WHERE metric_flag = TRUE"
c_project_ecoregion = pl.from_pandas(pd.read_sql(query, engine))

# Load c_project_model_group
query = f"SELECT * FROM {PROJECT_MODEL_GROUP_CROSSWALK}"
c_project_model_group = pl.from_pandas(pd.read_sql(query, engine))

# Load a_project
query = f"SELECT * FROM {A_PROJECT}"
a_project = pl.from_pandas(pd.read_sql(query, engine))

display(a_ecoregion.head())
display(c_project_ecoregion.head())
display(c_project_model_group.head())
display(a_project.head())

ecoregion_uid,geom,ecoregion_code,ecoregion_name,region_uid,ecoregion_description,ecoregion_id
i64,str,str,str,f64,str,i64
2,"""010600002023690000010000000103…","""LBAnw""","""Lower Barataria - northwest""",2.0,"""Barataria Basin south of GIWW …",12
3,"""010600002023690000010000000103…","""LBAse""","""Lower Barataria - southeast""",2.0,"""Barataria Basin from the Missi…",17
4,"""010600002023690000020000000103…","""LBAsw""","""Lower Barataria - southwest""",2.0,"""Barataria Basin between Bayou …",15
5,"""010600002023690000020000000103…","""MBA""","""Mid Barataria""",2.0,"""Middle Barataria Basin from US…",11
7,"""010600002023690000010000000103…","""ATB""","""Atchafalaya Basin""",1.0,"""Atchafalaya River basin upstre…",26


project_ecoregion_uid,project_id,ecoregion_uid,metric_flag,ecoregion_id
i64,i64,i64,bool,i64
1,60000,14,true,9
2,60000,17,true,7
3,60000,20,true,6
4,130100,1,true,13
5,130100,2,true,12


project_model_group_uid,project_id,model_group_id,cost_model_group_id,implementation_period,model_implementation,icm_flag
i64,i64,i64,i64,i64,str,bool
505,3490000,521,626,1,"""[Implemented in 2025/ICM Year …",true
506,3490000,626,626,1,"""[Implemented in 2025/ICM Year …",true
507,3500000,522,627,1,"""Candidates not chosen in Maste…",false
508,3500000,627,627,1,"""[Implemented in 2022/ICM Year …",true
509,3510000,522,627,1,"""Candidates not chosen in Maste…",false


project_id,project_number,project_version,display_id,project_name,alternate_name,project_type_code,candidate_flag,region_name,duration_construction_year,duration_ped_year,solicitation_source,project_description,legacy_project_number
i64,i64,str,str,str,str,str,bool,str,f64,f64,str,str,str
510000,51,null,"""051""","""Lake Pontchartrain Buffer Mars…",null,"""MC""",false,"""Pontchartrain""",null,null,"""Unknown""","""Marsh Buffer in front of lake …","""001.MC.19"""
520000,52,null,"""052""","""Pass a Loutre Marsh Creation""",null,"""MC""",false,"""Pontchartrain""",null,null,"""Unknown""","""Pass a Loutre Marsh Creation""","""001.MC.23"""
530000,53,null,"""053""","""Biloxi Marsh Oyster Reef""",null,"""OR""",false,"""Pontchartrain""",null,null,"""St. Bernard Parish Master Plan""","""Construct a living breakwater …","""001.OR.01a"""
550000,55,null,"""055""","""Manchac Landbridge Shoreline P…",null,"""SP""",false,"""Pontchartrain""",null,null,"""CIAP""","""Shoreline protection through r…","""001.SP.01"""
560000,56,null,"""056""","""Maurepas Shoreline Protection """,null,"""SP""",false,"""Pontchartrain""",null,null,"""Unknown""","""assume ave. H20 depth of 4 ft.…","""001.SP.02"""


In [7]:
# Project model groups only, excluding FWOA
project_model_group_ids = (
    c_project_model_group
    .filter(pl.col("icm_flag") == True)
    .filter(pl.col("model_group_id") != fwoa_model_group)
    .select("model_group_id")
    .unique()
    .get_column("model_group_id")
    .to_list()
)

# Include FWOA only for baseline comparison
model_group_id_list = project_model_group_ids + [fwoa_model_group]

print("Number of project model groups:", len(project_model_group_ids))
print("Total model groups including FWOA:", len(model_group_id_list))

Number of project model groups: 94
Total model groups including FWOA: 95


In [8]:
from sqlalchemy import text

with engine.connect() as conn:
    ecoregion_to_region = pl.from_pandas(pd.read_sql(
        text("""
            SELECT
                e.ecoregion_id,
                e.ecoregion_uid,
                e.ecoregion_code,
                e.ecoregion_name,
                r.region_uid,
                r.region_name,
                r.region_code
            FROM gsd.a_ecoregion e
            LEFT JOIN gsd.a_region r
                ON e.region_uid = r.region_uid
        """),
        conn
    ))

In [10]:
filtered_project_model_group = (
    c_project_model_group
    .filter(
        pl.col("model_group_id").is_in(project_model_group_ids)
    )
    .select([
        "project_id",
        "model_group_id",
        "implementation_period",
    ])
    .join(
        c_project_ecoregion
        .select([
            "project_id",
            "ecoregion_id",
        ])
        .unique(),
        on="project_id",
        how="inner",
    )
    .select([
        "project_id",
        "model_group_id",
        "ecoregion_id",
        "implementation_period",
    ])
    .unique()
)

display(filtered_project_model_group.head())

project_id,model_group_id,ecoregion_id,implementation_period
i64,i64,i64,i64
3220000,516,2,2
3070200,686,4,2
890100,658,12,2
390000,627,3,1
2980200,639,32,1


In [11]:
filtered_project_model_group.select([
    pl.col("project_id").n_unique().alias("n_projects"),
    pl.col("model_group_id").n_unique().alias("n_model_groups"),
    pl.col("ecoregion_id").n_unique().alias("n_ecoregions"),
])

n_projects,n_model_groups,n_ecoregions
u32,u32,u32
146,94,23


In [12]:
# Build all scenario/model-group combinations that will be processed.

scenario_model_group = list(itertools.product(scenario_list, model_group_id_list))
display(scenario_model_group)

# For each (scenario, model_group), fetch annual land-type raster paths 
# read_data query returns tif paths to the 50 years of model data in the form of a lazyframe 

scenario_model_group_dict = {
    (scenario, model_group_id): read_data(
        variable="lnd_type",
        grid="morph_pixel_v001",
        time_unit="annual",
        model_group_id=model_group_id,
        scenario_id=scenario,
    ).select(["calendar_year", "path"])
    for scenario, model_group_id in scenario_model_group
}

print(scenario_model_group_dict)


# structure =  {
#     (scenario, model_group_id) : pl.LazyFrame
# }

# Materialize year->path dict per (scenario, model_group) from lazyframes to the appropriate scenario model group for selection during windowed analysis 
# Downstream raster loops need a year -> filepath mapping for windowed reads.

tif_by_modelgroup_scenario_year = {
    my_tuple: dict(select_lazyframe.collect().iter_rows())
    for my_tuple, select_lazyframe in scenario_model_group_dict.items()
}

# structure =  {
#     (scenario, model_group_id) : {year : tif_path}
# }


[(7, 676),
 (7, 677),
 (7, 633),
 (7, 626),
 (7, 642),
 (7, 670),
 (7, 682),
 (7, 609),
 (7, 610),
 (7, 681),
 (7, 675),
 (7, 658),
 (7, 637),
 (7, 645),
 (7, 680),
 (7, 661),
 (7, 521),
 (7, 656),
 (7, 673),
 (7, 635),
 (7, 657),
 (7, 666),
 (7, 685),
 (7, 640),
 (7, 665),
 (7, 636),
 (7, 669),
 (7, 649),
 (7, 687),
 (7, 607),
 (7, 617),
 (7, 684),
 (7, 632),
 (7, 614),
 (7, 668),
 (7, 606),
 (7, 601),
 (7, 646),
 (7, 611),
 (7, 652),
 (7, 630),
 (7, 622),
 (7, 515),
 (7, 686),
 (7, 631),
 (7, 624),
 (7, 678),
 (7, 638),
 (7, 674),
 (7, 639),
 (7, 627),
 (7, 602),
 (7, 662),
 (7, 625),
 (7, 619),
 (7, 689),
 (7, 654),
 (7, 659),
 (7, 643),
 (7, 683),
 (7, 612),
 (7, 688),
 (7, 604),
 (7, 623),
 (7, 603),
 (7, 690),
 (7, 621),
 (7, 613),
 (7, 600),
 (7, 615),
 (7, 679),
 (7, 516),
 (7, 634),
 (7, 655),
 (7, 628),
 (7, 644),
 (7, 648),
 (7, 616),
 (7, 651),
 (7, 608),
 (7, 664),
 (7, 618),
 (7, 647),
 (7, 653),
 (7, 641),
 (7, 605),
 (7, 620),
 (7, 650),
 (7, 672),
 (7, 663),
 (7, 629),

{(7, 676): <LazyFrame at 0x7F93ED2BEAE0>, (7, 677): <LazyFrame at 0x7F93F589BCB0>, (7, 633): <LazyFrame at 0x7F93EC716F00>, (7, 626): <LazyFrame at 0x7F93EC714A10>, (7, 642): <LazyFrame at 0x7F93EC714980>, (7, 670): <LazyFrame at 0x7F93EC7172C0>, (7, 682): <LazyFrame at 0x7F93EC714AD0>, (7, 609): <LazyFrame at 0x7F93ED1D0350>, (7, 610): <LazyFrame at 0x7F93ED1D0590>, (7, 681): <LazyFrame at 0x7F93ED1D0170>, (7, 675): <LazyFrame at 0x7F93EC75B5F0>, (7, 658): <LazyFrame at 0x7F93EC75AA20>, (7, 637): <LazyFrame at 0x7F93EC758080>, (7, 645): <LazyFrame at 0x7F93EC758110>, (7, 680): <LazyFrame at 0x7F93EC7580B0>, (7, 661): <LazyFrame at 0x7F93EC75B080>, (7, 521): <LazyFrame at 0x7F93EC75B7A0>, (7, 656): <LazyFrame at 0x7F93EC75B620>, (7, 673): <LazyFrame at 0x7F93EC75AAB0>, (7, 635): <LazyFrame at 0x7F93EC75AA50>, (7, 657): <LazyFrame at 0x7F93EC75BCB0>, (7, 666): <LazyFrame at 0x7F93EC75BAA0>, (7, 685): <LazyFrame at 0x7F93EC759B80>, (7, 640): <LazyFrame at 0x7F93EC759E20>, (7, 665): <Lazy

In [13]:
# Build distinct (model_group, project, ecoregion) mappings from the filtered crosswalk.
# Downstream raster counting needs valid ecoregions associated with each model group.

subset_project_model_ecoregion_df = filtered_project_model_group.unique(
    subset=["model_group_id", "project_id", "ecoregion_id"]
).select(["model_group_id", "project_id", "ecoregion_id"])

# Collapse to one row per model group with a list of ecoregion IDs.
# This structure supports per-model-group masking during raster window loops.
project_model_ecoregion_df = subset_project_model_ecoregion_df.group_by(
    "model_group_id", maintain_order=True
).agg(col("ecoregion_id"))

project_model_ecoregion_df

model_group_id,ecoregion_id
i64,list[i64]
516,"[15, 32, … 11]"
521,"[24, 2, … 4]"
663,"[17, 9, … 10]"
668,"[7, 21, … 10]"
652,"[13, 10, … 17]"
…,…
634,"[21, 24, … 6]"
649,"[7, 3, … 4]"
670,"[17, 3, 29]"


## Calculate regional land-building benefit from land-type rasters

This section calculates land-building benefit from annual `lnd_type` rasters on the `morph_pixel_v001` grid. Land pixels are counted by scenario, model group, year, and ecoregion using the `morph_pixel_v001__ecoregion_v001` crosswalk.

For each ecoregion and year, land benefit is calculated as:

`project land area − FWOA land area`

Ecoregions are then mapped to regions, and results are assigned to projects and implementation periods using the project-model group crosswalk.

Final aggregation level:

`project_id × ScenarioID × Region × implementation_period`

The current selected unit is square meters.

Derived variables:

- `avg_annual_land_building`: mean annual regional land benefit
- `land_building_year50`: sum of annual regional land benefit across modeled years
- `n_years_land_building_positive`: number of years with positive regional land benefit

### Raster Window Analysis
Windowed raster reads and per-ecoregion land-pixel aggregation, followed by area/benefit calculations.

In [14]:
t1 = time.time()

# Determine what ecoregions are allowed for each model group and store in a dictionary of ecoregion id arrays by model group id 

ecoregion_by_model_group_dict = {}
all_ecoregion_ids = np.unique(project_model_ecoregion_df.get_column("ecoregion_id").explode())

for model_group in model_group_id_list:
    # FWOA uses all mapped ecoregions; project model groups use only their mapped subset.
    if model_group == fwoa_model_group:
        ecoregion_by_model_group_dict[model_group] = all_ecoregion_ids
    else:
        ecoregion_by_model_group_dict[model_group] = np.array(
            project_model_ecoregion_df
            .filter(col("model_group_id") == model_group)
            .get_column("ecoregion_id")
            .item()
        )

# Accumulate windows and the common ecoregion ids found per model group

window_by_model_group_dict = defaultdict(list)

with rio.open(MORPH_TO_ECOREGION_CROSSWALK) as index_src:
    # Pixel area in square meters from raster resolution.
    pixel_width, pixel_height = index_src.res
    pixel_area = abs(pixel_width * pixel_height)

    for block_index, window in index_src.block_windows(1):
        ecoregion_ids = index_src.read(1, window=window)

        for model_group_id in model_group_id_list:
            ecoregion_array = ecoregion_by_model_group_dict[model_group_id]
            mask = np.isin(ecoregion_ids, ecoregion_array)
            if not mask.any():
                continue
            
            common_ecoregion_ids = np.where(mask, ecoregion_ids, -9999) # -9999 is the mask sentinel value

            window_by_model_group_dict[model_group_id].append((window, common_ecoregion_ids))


# Accumulate land-pixel counts keyed by (scenario, model_group, year, ecoregion).

pixel_counts = defaultdict(int)

for (scenario, model_group_id), lnd_type_dict in tif_by_modelgroup_scenario_year.items():

    model_group_windows = window_by_model_group_dict[model_group_id]

    # print(f"For model group {model_group_id} and scenario {scenario}, the ecoregions are {ecoregion_array}")

    with ExitStack() as stack:
        year_src = {year: stack.enter_context(rio.open(tif))
                    for year, tif in lnd_type_dict.items()}
        
        for window, common_ecoregion_ids in model_group_windows:
            # For each year, count land pixels (excluding water=2 and nodata=-9999).
            for year, src in year_src.items():
                # with rio.open(src) as land_type_values:
                land_type_window = src.read(1, window=window)
                land_mask = (land_type_window != 2) & (land_type_window != -9999)
                land_ecoregion_pixels = common_ecoregion_ids[land_mask]

                valid_land_ecoregion_pixels = land_ecoregion_pixels[land_ecoregion_pixels != -9999]
                if valid_land_ecoregion_pixels.size == 0:
                    continue

                unique_ids, counts = np.unique(valid_land_ecoregion_pixels, return_counts=True)

                # Aggregate counts into the scenario/model_group/year/ecoregion key.
                for eco_id, count in zip(unique_ids, counts):
                        key = (scenario, model_group_id, int(year), int(eco_id))
                        pixel_counts[key] += int(count)

# Convert accumulator dict into row records for DataFrame creation in the next cell.
records = [
    {
        'scenario':k[0],
        'model_group_id':k[1],
        'year':k[2],
        'ecoregion_id':k[3],
        'land_pixel_count':v,
    }
    for k,v in pixel_counts.items()
]

t2 = time.time()
 
print(f"Total Run Time for Windowed Analysis: {t2 - t1}")

Total Run Time for Windowed Analysis: 1904.4231808185577


In [15]:
# Convert aggregated pixel-count records into a tabular frame for downstream joins/calculations.
land_area_df = pl.DataFrame(records)

# Split project runs (FWA) from the baseline run (FWOA model group 520).
fwp_land_area = land_area_df.filter(col("model_group_id") != fwoa_model_group)
fwoa_land_area = land_area_df.filter(col("model_group_id") == fwoa_model_group)

# Attach baseline counts by matching on shared dimensions (ecoregion, year, scenario).
joined_land_area = fwp_land_area.join(
    fwoa_land_area, 
    on=["ecoregion_id", "year", "scenario"], how="left", suffix="_FWOA"
)

# Map ecoregion/model-group totals back to projects while retaining ecoregion_id
joined_df = (
    joined_land_area
    .join(
        filtered_project_model_group,
        on=["ecoregion_id", "model_group_id"],
        how="inner",
        suffix="_crosswalk",
    )
    .group_by([
        "project_id",
        "scenario",
        "model_group_id",
        "ecoregion_id",
        "implementation_period",
        "year",
    ])
    .agg([
        pl.col("land_pixel_count").sum(),
        pl.col("land_pixel_count_FWOA").sum(),
    ])
    .sort(["project_id", "scenario", "ecoregion_id", "year"])
)

# Benefit area = project scenario land area minus baseline land area (in native square meters).
difference_area_exp = (
    (col("land_pixel_count") - col("land_pixel_count_FWOA")) * pixel_area
)

# Validate units; if invalid, default to square meters to avoid bad conversions.
if selected_unit not in valid_units:
    print(
        f"'{selected_unit}' is not valid, falling back to square meters. Valid options: {sorted(valid_units)}"
    )
    selected_unit = 'sq_meters'

difference_area_exp = (
    (col("land_pixel_count") - col("land_pixel_count_FWOA")) * pixel_area
) / conversion_factor

# Dynamic output column names aligned to the selected unit.
col_name = "project_benefit_acres" if selected_unit == "acres" else "project_benefit_sq_meters"

# Total project land area in the selected unit.
land_area_exp = col('land_pixel_count') * pixel_area / conversion_factor
land_col_name = "project_land_area_acres" if selected_unit == 'acres' else "project_land_area_sq_meters"

# Add both derived metrics to the project-level results table.
joined_df = joined_df.with_columns(
    difference_area_exp.alias(col_name),
    land_area_exp.alias(land_col_name)
)

In [16]:
joined_df

project_id,scenario,model_group_id,ecoregion_id,implementation_period,year,land_pixel_count,land_pixel_count_FWOA,project_benefit_sq_meters,project_land_area_sq_meters
i64,i64,i64,i64,i64,i64,i64,i64,f64,f64
60000,7,601,6,1,2019,671508,716523,-4.05135e7,6.043572e8
60000,7,655,6,2,2020,605572,716506,-9.98406e7,5.450148e8
60000,7,601,6,1,2020,671436,716506,-4.0563e7,6.042924e8
60000,7,601,6,1,2021,660260,711278,-4.59162e7,5.94234e8
60000,7,601,6,1,2022,659395,710911,-4.63644e7,5.934555e8
…,…,…,…,…,…,…,…,…,…
3620000,8,521,28,1,2066,341036,291890,4.42314e7,3.069324e8
3620000,8,521,28,1,2067,292431,272268,1.81467e7,2.631879e8
3620000,8,521,28,1,2068,245511,245279,208800.0,2.209599e8


### Summarize annual land benefit to project-scenario-region level

This section maps each ecoregion to its corresponding region and sums annual land benefit across the ecoregions within each project-region combination.

Annual land benefit is stored as `land_building_sq_meters` and summarized at:

`project_id × ScenarioID × Region × implementation_period`

The summary calculates average annual land benefit, cumulative benefit across modeled years, and the number of years with positive land benefit.

In [17]:
land_project_annual_benefit = (
    joined_df
    .join(
        ecoregion_to_region.select([
            "ecoregion_id",
            "region_name",
        ]),
        on="ecoregion_id",
        how="left",
    )
    .rename({
        "scenario": "ScenarioID",
        "region_name": "Region",
        "project_benefit_sq_meters": "land_building_sq_meters",
        "project_land_area_sq_meters": "project_land_area_sq_meters",
    })
    .group_by([
        "project_id",
        "ScenarioID",
        "Region",
        "model_group_id",
        "implementation_period",
        "year",
    ])
    .agg([
        pl.col("land_building_sq_meters")
        .sum()
        .alias("land_building_sq_meters"),

        pl.col("project_land_area_sq_meters")
        .sum()
        .alias("project_land_area_sq_meters"),
    ])
)

In [18]:
land_project_summary = (
    land_project_annual_benefit
    .group_by([
        "project_id",
        "ScenarioID",
        "Region",
        "implementation_period",
    ])
    .agg([
        pl.col("land_building_sq_meters")
        .mean()
        .alias("avg_annual_land_building"),

        pl.col("land_building_sq_meters")
        .sum()
        .alias("land_building_year50"),

        (pl.col("land_building_sq_meters") > 0)
        .sum()
        .alias("n_years_land_building_positive"),
    ])
)

## Attach project metadata

This section joins descriptive project information from `a_project` to the regional land-benefit summary.

Fields added:

- `BaseID` from `project_id`
- `ProjectName` from `project_name`
- `TypeCode` from `project_type_code`
- `display_id`

`Region` and `implementation_period` are already present in the benefit summary and are not taken from `a_project`.

In [19]:
project_lu = (
    a_project
    .select([
        "project_id",
        pl.col("project_id").alias("BaseID"),
        pl.col("project_name").alias("ProjectName"),
        pl.col("project_type_code").alias("TypeCode"),
        pl.col("display_id"),
    ])
    .unique()
)

land_project_summary = land_project_summary.join(
    project_lu,
    on="project_id",
    how="left",
)

In [20]:
land_project_summary

project_id,ScenarioID,Region,implementation_period,avg_annual_land_building,land_building_year50,n_years_land_building_positive,BaseID,ProjectName,TypeCode,display_id
i64,i64,str,i64,f64,f64,u32,i64,str,str,str
1080000,8,"""Terrebonne""",1,9.0700e7,1.4149e10,140,1080000,"""Atchafalaya River Diversion""","""DI""","""108"""
2860000,8,"""Terrebonne""",1,1.4748e7,7.668738e8,44,2860000,"""North Lake Mechant Marsh Creat…","""MC""","""286a"""
1390100,7,"""Barataria""",1,-5.1770e6,-2.6920e8,4,1390100,"""Increase Atchafalaya Flow to T…","""DI""","""139b"""
3330000,7,"""Barataria""",1,1.0119e6,5.26185e7,42,3330000,"""Three Ridge Restoration""","""RR""","""333"""
3110000,7,"""Pontchartrain""",1,-2.7519e7,-1.4310e9,8,3110000,"""Black and Eloi Bay Ridge and M…","""IP""","""311"""
…,…,…,…,…,…,…,…,…,…,…
3270000,7,"""Pontchartrain""",1,-1.7784e7,-9.2477e8,18,3270000,"""Lower Plaquemines River Sedime…","""IP""","""327"""
3220000,8,"""Barataria""",2,1.1484e8,1.5618e10,119,3220000,"""Freshwater Delivery to Western…","""DI""","""322"""
2450000,8,"""Pontchartrain""",2,-3.7578e7,-1.2025e9,12,2450000,"""LaBranche Hydrologic Restorati…","""HR""","""245"""


## Calculate regional HSI benefit from habitat suitability rasters

This section calculates Habitat Suitability Index benefit from annual `hsi` rasters on the `veg_grid_cell_v001` grid for the species codes `oyste` and `spsta`.

The `veg_grid_cell_v001__ecoregion_v001` crosswalk is used to assign HSI raster cells to project ecoregions. Ecoregions are then mapped to regions and implementation periods.

For each species, year, and project-region combination, HSI benefit is calculated as:

`project mean HSI − FWOA mean HSI`

Annual HSI benefits are then averaged across years.

Final aggregation level:

`project_id × ScenarioID × Region × implementation_period`

Derived variables:

- `HSI_OYSTE`: average annual oyster HSI benefit relative to FWOA
- `HSI_SPSTA`: average annual spotted seatrout HSI benefit relative to FWOA

### Load HSI raster paths

This section loads annual HSI raster paths for the selected scenarios, model groups, and species. HSI is stored on the `veg_grid_cell_v001` grid and requires lowercase species codes. The resulting table contains raster paths, not HSI values.

In [21]:
species_codes = ["oyste", "spsta"]

hsi_lazyframes = []

for scenario_id in scenario_list:
    for model_group_id in model_group_id_list:
        for species_code in species_codes:
            lf = read_data(
                variable="hsi",
                grid="veg_grid_cell_v001",
                time_unit="annual",
                model_group_id=model_group_id,
                scenario_id=scenario_id,
                species_code=species_code,
            )
            hsi_lazyframes.append(lf)

hsi_paths_df = pl.concat([
    lf.collect() for lf in hsi_lazyframes
])

display(hsi_paths_df.head())

variable,grid,time_unit,model_group_id,scenario_id,species_code,calendar_year,path
str,str,str,i32,i32,str,i32,str
"""hsi""","""veg_grid_cell_v001""","""annual""",633,7,"""oyste""",2021,"""/ocean/projects/bcs200002p/sha…"
"""hsi""","""veg_grid_cell_v001""","""annual""",633,7,"""oyste""",2022,"""/ocean/projects/bcs200002p/sha…"
"""hsi""","""veg_grid_cell_v001""","""annual""",633,7,"""oyste""",2024,"""/ocean/projects/bcs200002p/sha…"
"""hsi""","""veg_grid_cell_v001""","""annual""",633,7,"""oyste""",2025,"""/ocean/projects/bcs200002p/sha…"
"""hsi""","""veg_grid_cell_v001""","""annual""",633,7,"""oyste""",2028,"""/ocean/projects/bcs200002p/sha…"


### Prepare HSI ecoregion, region, and project lookup

This section prepares the spatial and project lookup information needed to calculate regional HSI benefit.

The `veg_grid_cell_v001__ecoregion_v001` crosswalk assigns each HSI raster cell to an ecoregion. The project-model group lookup identifies the project, model group, ecoregions, and implementation period associated with each model run. Ecoregions are mapped to `Region` using the ecoregion-region lookup.

In [22]:
import numpy as np
import rasterio as rio

VEG_TO_ECOREGION_CROSSWALK = (
    "/ocean/projects/bcs200002p/shared/grids/crosswalks/"
    "veg_grid_cell_v001__ecoregion_v001.tif"
)

project_ecoregion_lookup = (
    filtered_project_model_group
    .select([
        "project_id",
        "model_group_id",
        "ecoregion_id",
        "implementation_period",
    ])
    .unique()
    .join(
        ecoregion_to_region.select([
            "ecoregion_id",
            "region_name",
        ]),
        on="ecoregion_id",
        how="left",
    )
    .rename({
        "region_name": "Region",
    })
)

with rio.open(VEG_TO_ECOREGION_CROSSWALK) as eco_src:
    eco_arr = eco_src.read(1)

### Calculate annual regional HSI benefit

This section matches each project HSI raster to the corresponding FWOA raster using scenario, species, and year.

For each project, region, and implementation period, mean project HSI and mean FWOA HSI are calculated over the project’s mapped ecoregions. Annual HSI benefit is calculated as:

`project_hsi_mean − fwoa_hsi_mean`

The resulting table retains project, scenario, region, implementation period, model group, species, and year.

In [23]:
# Separate project HSI rasters from FWOA HSI rasters
project_hsi_paths = (
    hsi_paths_df
    .filter(
        pl.col("model_group_id") != fwoa_model_group
    )
)

fwoa_hsi_paths = (
    hsi_paths_df
    .filter(
        pl.col("model_group_id") == fwoa_model_group
    )
    .select([
        "scenario_id",
        "species_code",
        "calendar_year",
        pl.col("path").alias("fwoa_path"),
    ])
)

# Match each project raster with the corresponding FWOA raster
project_hsi_paths = (
    project_hsi_paths
    .join(
        fwoa_hsi_paths,
        on=[
            "scenario_id",
            "species_code",
            "calendar_year",
        ],
        how="left",
    )
)

hsi_records = []

for row in project_hsi_paths.iter_rows(named=True):

    scenario_id = row["scenario_id"]
    model_group_id = row["model_group_id"]
    species_code = row["species_code"]
    year = row["calendar_year"]

    project_hsi_path = row["path"]
    fwoa_hsi_path = row["fwoa_path"]

    if fwoa_hsi_path is None:
        continue

    project_ecos_this_model = (
        project_ecoregion_lookup
        .filter(
            pl.col("model_group_id") == model_group_id
        )
    )

    if project_ecos_this_model.height == 0:
        continue

    with rio.open(project_hsi_path) as project_src:
        project_hsi_arr = project_src.read(1)
        project_nodata = project_src.nodata

    with rio.open(fwoa_hsi_path) as fwoa_src:
        fwoa_hsi_arr = fwoa_src.read(1)
        fwoa_nodata = fwoa_src.nodata

    if project_hsi_arr.shape != fwoa_hsi_arr.shape:
        raise ValueError(
            f"Project and FWOA HSI raster shapes differ: "
            f"scenario={scenario_id}, "
            f"model_group={model_group_id}, "
            f"species={species_code}, "
            f"year={year}"
        )

    if project_hsi_arr.shape != eco_arr.shape:
        raise ValueError(
            f"HSI raster and ecoregion crosswalk shapes differ: "
            f"scenario={scenario_id}, "
            f"model_group={model_group_id}, "
            f"species={species_code}, "
            f"year={year}"
        )

    project_region_pairs = (
        project_ecos_this_model
        .select([
            "project_id",
            "Region",
            "implementation_period",
        ])
        .unique()
    )

    for project_region in project_region_pairs.iter_rows(named=True):

        project_id = project_region["project_id"]
        region = project_region["Region"]
        implementation_period = project_region["implementation_period"]
        eco_ids = (
            project_ecos_this_model
            .filter(
                (pl.col("project_id") == project_id)
                & (pl.col("Region") == region)
                & (pl.col("implementation_period") == implementation_period)
            )
            .get_column("ecoregion_id")
            .to_numpy()
        )

        mask = np.isin(eco_arr, eco_ids)

        if project_nodata is not None:
            mask = mask & (
                project_hsi_arr != project_nodata
            )

        if fwoa_nodata is not None:
            mask = mask & (
                fwoa_hsi_arr != fwoa_nodata
            )

        mask = (
            mask
            & np.isfinite(project_hsi_arr)
            & np.isfinite(fwoa_hsi_arr)
        )

        if not mask.any():
            continue

        project_hsi_mean = float(
            np.mean(project_hsi_arr[mask])
        )

        fwoa_hsi_mean = float(
            np.mean(fwoa_hsi_arr[mask])
        )

        hsi_records.append({
            "project_id": project_id,
            "ScenarioID": scenario_id,
            "Region": region,
            "implementation_period": implementation_period,
            "model_group_id": model_group_id,
            "species_code": species_code,
            "year": year,
            "project_hsi_mean": project_hsi_mean,
            "fwoa_hsi_mean": fwoa_hsi_mean,
            "hsi_benefit": (
                project_hsi_mean - fwoa_hsi_mean
            ),
        })

hsi_annual_project = pl.DataFrame(hsi_records)

display(hsi_annual_project.head())

project_id,ScenarioID,Region,implementation_period,model_group_id,species_code,year,project_hsi_mean,fwoa_hsi_mean,hsi_benefit
i64,i64,str,i64,i64,str,i64,f64,f64,f64
2510000,7,"""Pontchartrain""",1,633,"""oyste""",2021,0.0,0.0,0.0
2880000,7,"""Central Coast""",1,633,"""oyste""",2021,0.077309,0.077218,0.000091
2880000,7,"""Chenier Plain""",1,633,"""oyste""",2021,0.88728,0.883809,0.003471
3350100,7,"""Terrebonne""",1,633,"""oyste""",2021,0.191199,0.188352,0.002847
3350100,7,"""Terrebonne""",1,633,"""oyste""",2022,0.235412,0.230856,0.004556


### Summarize HSI to project-scenario-region level

This section averages annual HSI benefit across years for each species and pivots the species results into separate columns.

Final aggregation level:

`project_id × ScenarioID × Region × implementation_period`

The resulting columns are `HSI_OYSTE` and `HSI_SPSTA`.

In [24]:
hsi_project_summary_long = (
    hsi_annual_project
    .group_by([
        "project_id",
        "ScenarioID",
        "Region",
        "implementation_period",
        "species_code",
    ])
    .agg([
        pl.col("hsi_benefit")
        .mean()
        .alias("HSI"),

        pl.col("project_hsi_mean")
        .mean()
        .alias("project_hsi_mean"),

        pl.col("fwoa_hsi_mean")
        .mean()
        .alias("fwoa_hsi_mean"),
    ])
)

hsi_project_summary = (
    hsi_project_summary_long
    .with_columns(
        pl.when(
            pl.col("species_code") == "oyste"
        )
        .then(
            pl.lit("HSI_OYSTE")
        )
        .when(
            pl.col("species_code") == "spsta"
        )
        .then(
            pl.lit("HSI_SPSTA")
        )
        .otherwise(
            pl.col("species_code")
        )
        .alias("hsi_column")
    )
    .pivot(
        values="HSI",
        index=[
            "project_id",
            "ScenarioID",
            "Region",
            "implementation_period",
        ],
        on="hsi_column",
    )
)

display(hsi_project_summary)

project_id,ScenarioID,Region,implementation_period,HSI_SPSTA,HSI_OYSTE
i64,i64,str,i64,f64,f64
3120000,7,"""Pontchartrain""",1,-0.069768,-0.07412
540000,7,"""Pontchartrain""",2,-0.022009,0.012155
1570200,7,"""Central Coast""",1,0.004349,0.000063
2710000,8,"""Barataria""",1,-0.075108,0.00359
2840000,8,"""Barataria""",1,-0.006641,0.043881
…,…,…,…,…,…
3000200,8,"""Chenier Plain""",1,-0.001264,-0.000898
3150000,8,"""Pontchartrain""",2,0.000693,0.0
3320000,7,"""Barataria""",1,-0.005414,-0.015916


### Join HSI with the land-benefit summary

This section joins the regional HSI benefit summary to the regional land-benefit summary using:

`project_id × ScenarioID × Region × implementation_period`

In [25]:
land_plus_hsi = (
    land_project_summary
    .join(
        hsi_project_summary,
        on=[
            "project_id",
            "ScenarioID",
            "Region",
            "implementation_period",
        ],
        how="left",
    )
)

display(land_plus_hsi)

project_id,ScenarioID,Region,implementation_period,avg_annual_land_building,land_building_year50,n_years_land_building_positive,BaseID,ProjectName,TypeCode,display_id,HSI_SPSTA,HSI_OYSTE
i64,i64,str,i64,f64,f64,u32,i64,str,str,str,f64,f64
1080000,8,"""Terrebonne""",1,9.0700e7,1.4149e10,140,1080000,"""Atchafalaya River Diversion""","""DI""","""108""",-0.011029,-0.008383
2860000,8,"""Terrebonne""",1,1.4748e7,7.668738e8,44,2860000,"""North Lake Mechant Marsh Creat…","""MC""","""286a""",0.001199,0.0
1390100,7,"""Barataria""",1,-5.1770e6,-2.6920e8,4,1390100,"""Increase Atchafalaya Flow to T…","""DI""","""139b""",null,null
3330000,7,"""Barataria""",1,1.0119e6,5.26185e7,42,3330000,"""Three Ridge Restoration""","""RR""","""333""",-0.022995,-0.017725
3110000,7,"""Pontchartrain""",1,-2.7519e7,-1.4310e9,8,3110000,"""Black and Eloi Bay Ridge and M…","""IP""","""311""",-0.003422,0.025917
…,…,…,…,…,…,…,…,…,…,…,…,…
3270000,7,"""Pontchartrain""",1,-1.7784e7,-9.2477e8,18,3270000,"""Lower Plaquemines River Sedime…","""IP""","""327""",0.002059,0.0
3220000,8,"""Barataria""",2,1.1484e8,1.5618e10,119,3220000,"""Freshwater Delivery to Western…","""DI""","""322""",-0.004518,-0.002076
2450000,8,"""Pontchartrain""",2,-3.7578e7,-1.2025e9,12,2450000,"""LaBranche Hydrologic Restorati…","""HR""","""245""",null,null


## Calculate project cost metrics

This section calculates project cost metrics from `pct.vw_o_cost_project`.

Costs are restricted to the projects and scenarios included in the benefit dataset and to `cost_scenario_id = 2`. All available implementation periods are retained.

Cost aggregation level:

`project_id × ScenarioID × implementation_period`

Derived variables:

- `cost_total_min`: minimum available `total_cost` within the selected cost scenario
- `cost_total_max`: maximum available `total_cost` within the selected cost scenario

Cost is not region-specific in the source table. When joined to the regional benefit table, the same project-scenario-period cost is assigned to each applicable regional row.

### Inspect project cost source

This section briefly inspects `pct.vw_o_cost_project`, the project-level cost view used to calculate cost metrics before they are joined to the regional benefit records.

In [26]:
from sqlalchemy import text
import pandas as pd
import polars as pl

with engine.connect() as conn:
    cost_project_sample = pd.read_sql(
        text("""
            SELECT *
            FROM pct.vw_o_cost_project
            LIMIT 10
        """),
        conn
    )

display(cost_project_sample)

,project_id,display_id,project_type_code,model_group_id,cost_model_group_id,implementation_period,scenario_id,cost_scenario_id,start_year,ped_cost,om_cost_year,cm_cost,contingency_cost,construction_cost,survey_cost,mobilization_cost,component_cost,om_cost_total,total_cost
0,60000,006,DI,522,601,1,7,1,2028,2.642401e+07,1.321200e+06,1.321200e+07,5.284801e+07,2.642401e+08,6.137980e+06,1.258286e+07,2.455192e+08,5.549041e+07,3.990025e+08
1,60000,006,DI,522,601,1,7,2,2028,3.387693e+07,1.693847e+06,1.693847e+07,6.775386e+07,3.387693e+08,7.869206e+06,1.613187e+07,3.147682e+08,7.114155e+07,5.115416e+08
2,60000,006,DI,522,601,1,7,3,2028,4.132986e+07,2.066493e+06,2.066493e+07,8.265971e+07,4.132986e+08,9.600431e+06,1.968088e+07,3.840172e+08,8.679270e+07,6.240808e+08
3,60000,006,DI,522,601,1,8,1,2028,2.642401e+07,1.321200e+06,1.321200e+07,5.284801e+07,2.642401e+08,6.137980e+06,1.258286e+07,2.455192e+08,5.549041e+07,3.990025e+08
4,60000,006,DI,522,601,1,8,2,2028,3.387693e+07,1.693847e+06,1.693847e+07,6.775386e+07,3.387693e+08,7.869206e+06,1.613187e+07,3.147682e+08,7.114155e+07,5.115416e+08
5,60000,006,DI,522,601,1,8,3,2028,4.132986e+07,2.066493e+06,2.066493e+07,8.265971e+07,4.132986e+08,9.600431e+06,1.968088e+07,3.840172e+08,8.679270e+07,6.240808e+08
6,130100,013b,DI,522,614,1,7,1,2030,1.027391e+08,1.016830e+06,5.136956e+07,2.054782e+08,1.027391e+09,2.386507e+07,4.892339e+07,9.546027e+08,4.067318e+07,1.376282e+09
7,130100,013b,DI,522,614,1,7,2,2030,1.198098e+08,1.186548e+06,5.990489e+07,2.396196e+08,1.198098e+09,2.783038e+07,5.705228e+07,1.113215e+09,4.746190e+07,1.604989e+09
8,130100,013b,DI,522,614,1,7,3,2030,1.368805e+08,1.356266e+06,6.844023e+07,2.737609e+08,1.368805e+09,3.179569e+07,6.518117e+07,1.271828e+09,5.425062e+07,1.833697e+09
9,130100,013b,DI,522,614,1,8,1,2030,1.027391e+08,1.016830e+06,5.136956e+07,2.054782e+08,1.027391e+09,2.386507e+07,4.892339e+07,9.546027e+08,4.067318e+07,1.376282e+09


### Load project cost data

This section loads the cost fields needed from `pct.vw_o_cost_project`. The table includes project, scenario, implementation period, cost scenario, and total cost information.

In [27]:
with engine.connect() as conn:
    cost_raw = pl.from_pandas(pd.read_sql(
        text("""
            SELECT
                project_id,
                display_id,
                project_type_code,
                model_group_id,
                cost_model_group_id,
                implementation_period,
                scenario_id,
                cost_scenario_id,
                start_year,
                total_cost
            FROM pct.vw_o_cost_project
        """),
        conn
    ))

display(cost_raw.head())

project_id,display_id,project_type_code,model_group_id,cost_model_group_id,implementation_period,scenario_id,cost_scenario_id,start_year,total_cost
i64,str,str,i64,i64,i64,i64,i64,i64,f64
60000,"""006""","""DI""",522,601,1,7,1,2028,3.9900e8
60000,"""006""","""DI""",522,601,1,7,2,2028,5.1154e8
60000,"""006""","""DI""",522,601,1,7,3,2028,6.2408e8
60000,"""006""","""DI""",522,601,1,8,1,2028,3.9900e8
60000,"""006""","""DI""",522,601,1,8,2,2028,5.1154e8


### Filter cost data to selected projects and scenarios

This section restricts the cost table to the projects and scenarios included in the land and HSI summary.

Only `cost_scenario_id = 2` is retained. No implementation-period filter is applied, so costs for all available implementation periods are included.

In [28]:
project_ids_needed = (
    land_plus_hsi
    .get_column("project_id")
    .unique()
    .to_list()
)

scenarios_needed = (
    land_plus_hsi
    .get_column("ScenarioID")
    .unique()
    .to_list()
)

cost_filtered = (
    cost_raw
    .filter(pl.col("project_id").is_in(project_ids_needed))
    .filter(pl.col("scenario_id").is_in(scenarios_needed))
    .filter(pl.col("cost_scenario_id") == 2)
)

display(cost_filtered.head())

project_id,display_id,project_type_code,model_group_id,cost_model_group_id,implementation_period,scenario_id,cost_scenario_id,start_year,total_cost
i64,str,str,i64,i64,i64,i64,i64,i64,f64
60000,"""006""","""DI""",522,601,1,7,2,2028,5.1154e8
60000,"""006""","""DI""",522,601,1,8,2,2028,5.1154e8
130100,"""013b""","""DI""",522,614,1,7,2,2030,1.6050e9
130100,"""013b""","""DI""",522,614,1,8,2,2030,1.6050e9
140000,"""014a""","""DI""",521,663,2,7,2,2046,3.4370e8


### Summarize cost to project-scenario-implementation-period level

This section summarizes cost at:

`project_id × ScenarioID × implementation_period`

The minimum and maximum `total_cost` values are calculated across the available rows remaining after filtering to `cost_scenario_id = 2`. If only one cost record exists for a project-scenario-period combination, `cost_total_min` and `cost_total_max` will be equal.

In [29]:
cost_summary = (
    cost_filtered
    .group_by([
        "project_id",
        "scenario_id",
        "implementation_period",
    ])
    .agg([
        pl.col("total_cost").min().alias("cost_total_min"),
        pl.col("total_cost").max().alias("cost_total_max"),
    ])
    .rename({"scenario_id": "ScenarioID"})
)

display(cost_summary)

project_id,ScenarioID,implementation_period,cost_total_min,cost_total_max
i64,i64,i64,f64,f64
2470000,8,1,5.5784e7,5.5784e7
2670000,8,2,2.9687e8,2.9687e8
3530000,7,1,4.5286e8,4.5286e8
3470000,7,1,1.6336e8,1.6336e8
3260000,8,1,1.5217e9,1.5217e9
…,…,…,…,…
3620000,8,1,7.9435e8,7.9435e8
130100,7,1,1.6050e9,1.6050e9
1290000,7,1,1.7800e7,1.7800e7


### Join cost with land and HSI metrics

This section joins project cost to the combined regional land and HSI table using:

`project_id × ScenarioID × implementation_period`

Because the cost source is not regional, the matching project-scenario-period cost is repeated across the applicable regional rows.

In [30]:
land_plus_hsi_cost = (
    land_plus_hsi
    .join(
        cost_summary,
        on=[
            "project_id",
            "ScenarioID",
            "implementation_period",
        ],
        how="left"
    )
)

display(land_plus_hsi_cost)

project_id,ScenarioID,Region,implementation_period,avg_annual_land_building,land_building_year50,n_years_land_building_positive,BaseID,ProjectName,TypeCode,display_id,HSI_SPSTA,HSI_OYSTE,cost_total_min,cost_total_max
i64,i64,str,i64,f64,f64,u32,i64,str,str,str,f64,f64,f64,f64
1080000,8,"""Terrebonne""",1,9.0700e7,1.4149e10,140,1080000,"""Atchafalaya River Diversion""","""DI""","""108""",-0.011029,-0.008383,7.9435e8,7.9435e8
2860000,8,"""Terrebonne""",1,1.4748e7,7.668738e8,44,2860000,"""North Lake Mechant Marsh Creat…","""MC""","""286a""",0.001199,0.0,1.1436e9,1.1436e9
1390100,7,"""Barataria""",1,-5.1770e6,-2.6920e8,4,1390100,"""Increase Atchafalaya Flow to T…","""DI""","""139b""",null,null,5.9343e8,5.9343e8
3330000,7,"""Barataria""",1,1.0119e6,5.26185e7,42,3330000,"""Three Ridge Restoration""","""RR""","""333""",-0.022995,-0.017725,4.9135e7,4.9135e7
3110000,7,"""Pontchartrain""",1,-2.7519e7,-1.4310e9,8,3110000,"""Black and Eloi Bay Ridge and M…","""IP""","""311""",-0.003422,0.025917,7.6719e8,7.6719e8
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
3270000,7,"""Pontchartrain""",1,-1.7784e7,-9.2477e8,18,3270000,"""Lower Plaquemines River Sedime…","""IP""","""327""",0.002059,0.0,4.2403e9,4.2403e9
3220000,8,"""Barataria""",2,1.1484e8,1.5618e10,119,3220000,"""Freshwater Delivery to Western…","""DI""","""322""",-0.004518,-0.002076,1.4861e8,1.4861e8
2450000,8,"""Pontchartrain""",2,-3.7578e7,-1.2025e9,12,2450000,"""LaBranche Hydrologic Restorati…","""HR""","""245""",null,null,null,null


In [31]:

missing_cost_rows = land_plus_hsi_cost.filter(
    pl.col("cost_total_min").is_null() | pl.col("cost_total_max").is_null()
)

print(missing_cost_rows.shape)

display(
    missing_cost_rows
    .select(["project_id", "BaseID", "ScenarioID", "ProjectName", "TypeCode", "display_id", "Region", "implementation_period"])
    .unique()
    .sort(["project_id", "ScenarioID", "implementation_period"])
)

(161, 15)


project_id,BaseID,ScenarioID,ProjectName,TypeCode,display_id,Region,implementation_period
i64,i64,i64,str,str,str,str,i64
60000,60000,7,"""Lower Breton Diversion""","""DI""","""006""","""Pontchartrain""",2
60000,60000,8,"""Lower Breton Diversion""","""DI""","""006""","""Pontchartrain""",2
140000,140000,7,"""Central Wetlands Diversion""","""DI""","""014a""","""Pontchartrain""",1
140000,140000,7,"""Central Wetlands Diversion""","""DI""","""014a""","""Barataria""",1
140000,140000,8,"""Central Wetlands Diversion""","""DI""","""014a""","""Pontchartrain""",1
…,…,…,…,…,…,…,…
3520000,3520000,8,"""Wildhorse Marsh Creation""","""MC""","""352""","""Chenier Plain""",2
3530000,3530000,7,"""Chenier Ridges Restoration""","""RR""","""353""","""Chenier Plain""",2
3530000,3530000,8,"""Chenier Ridges Restoration""","""RR""","""353""","""Chenier Plain""",2


In [32]:
missing_project_ids = (
    missing_cost_rows
    .get_column("project_id")
    .unique()
    .to_list()
)

missing_project_ids_sql = ",".join(map(str, missing_project_ids))

with engine.connect() as conn:
    missing_cost_check = pl.from_pandas(pd.read_sql(
        text(f"""
            SELECT
                project_id,
                scenario_id,
                implementation_period,
                cost_scenario_id,
                COUNT(*) AS n_rows,
                MIN(total_cost) AS min_total_cost,
                MAX(total_cost) AS max_total_cost
            FROM pct.vw_o_cost_project
            WHERE project_id IN ({missing_project_ids_sql})
            GROUP BY
                project_id,
                scenario_id,
                implementation_period,
                cost_scenario_id
            ORDER BY
                project_id,
                scenario_id,
                implementation_period,
                cost_scenario_id
        """),
        conn
    ))

display(missing_cost_check)

project_id,scenario_id,implementation_period,cost_scenario_id,n_rows,min_total_cost,max_total_cost
i64,i64,i64,i64,i64,f64,f64
60000,7,1,1,1,3.9900e8,3.9900e8
60000,7,1,2,1,5.1154e8,5.1154e8
60000,7,1,3,1,6.2408e8,6.2408e8
60000,8,1,1,1,3.9900e8,3.9900e8
60000,8,1,2,1,5.1154e8,5.1154e8
…,…,…,…,…,…,…
3540000,7,1,2,1,7.7396e8,7.7396e8
3540000,7,1,3,1,8.4521e8,8.4521e8
3540000,8,1,1,1,7.9622e8,7.9622e8


### Check missing cost values

This check identifies project-scenario-region-implementation-period rows that did not receive matching cost values after the join.

In [33]:
land_plus_hsi_cost.select([
    pl.col("cost_total_min").null_count().alias("missing_cost_total_min"),
    pl.col("cost_total_max").null_count().alias("missing_cost_total_max"),
])

missing_cost_total_min,missing_cost_total_max
u32,u32
161,161


## Calculate regional fresh-marsh benefit

This section calculates incremental fresh-marsh benefit using `marsh_area_m2` from the annual vegetation-element output table (`icm.o_veg_annual_element`).

Each vegetation element is assigned to a region using the vegetation-element lookup (`cma.a_element`) and region lookup (`gsd.a_region`).

Fresh-marsh benefit is calculated as:

`project fresh-marsh area − FWOA fresh-marsh area`

Final aggregation level:

`project_id × ScenarioID × Region × implementation_period`

### Inspect and load annual marsh-area data

This section inspects and loads `marsh_area_m2` from the annual vegetation-element output table (`icm.o_veg_annual_element`) for the selected scenarios, project model groups, and FWOA model group.

Each vegetation element is assigned to `Region` using the vegetation-element lookup (`cma.a_element`) and region lookup (`gsd.a_region`). Rows without a region assignment are identified for review.

In [34]:
from sqlalchemy import text
import pandas as pd
import polars as pl

with engine.connect() as conn:
    veg_sample = pd.read_sql(
        text("""
            SELECT *
            FROM icm.o_veg_annual_element
            LIMIT 10
        """),
        conn
    )

display(veg_sample)

,element_id,model_group_id,scenario_id,calendar_year,marsh_area_m2,marsh_volume_m3,upload_date
0,370001,600,8,2049,23636700.0,10542326.0,2022-09-26 18:16:50.880
1,370001,600,7,2049,23886000.0,7468386.0,2022-09-26 18:16:50.880
2,370002,600,7,2049,24862500.0,8027668.0,2022-09-26 18:16:50.880
3,370002,600,8,2049,24489000.0,10597221.0,2022-09-26 18:16:50.880
4,370003,600,7,2049,29415600.0,9122912.0,2022-09-26 18:16:50.880
5,370003,600,8,2049,29203200.0,12807141.0,2022-09-26 18:16:50.880
6,370004,600,8,2049,37759500.0,17086436.0,2022-09-26 18:16:50.880
7,370004,600,7,2049,38055600.0,12411083.0,2022-09-26 18:16:50.880
8,370005,600,7,2049,8109900.0,2090302.0,2022-09-26 18:16:50.880
9,370005,600,8,2049,8045100.0,3115556.0,2022-09-26 18:16:50.880


In [35]:
model_groups_needed = project_model_group_ids + [fwoa_model_group]
scenarios_needed = scenario_list

model_groups_sql = ",".join(map(str, model_groups_needed))
scenarios_sql = ",".join(map(str, scenarios_needed))

with engine.connect() as conn:
    veg_marsh_raw = pl.from_pandas(
        pd.read_sql(
            text(f"""
                SELECT
                    v.element_id,
                    v.model_group_id,
                    v.scenario_id,
                    v.calendar_year,
                    v.marsh_area_m2,
                    r.region_name
                FROM icm.o_veg_annual_element v

                LEFT JOIN cma.a_element e
                    ON v.element_id = e.element_id

                LEFT JOIN gsd.a_region r
                    ON e.region_uid = r.region_uid

                WHERE v.model_group_id IN ({model_groups_sql})
                  AND v.scenario_id IN ({scenarios_sql})
            """),
            conn
        )
    ).rename({
        "region_name": "Region"
    })

print(veg_marsh_raw.shape)
display(veg_marsh_raw.head())

(484, 6)


element_id,model_group_id,scenario_id,calendar_year,marsh_area_m2,Region
i64,i64,i64,i64,f64,str
370001,600,8,2049,2.36367e7,"""Pontchartrain"""
370001,600,7,2049,2.3886e7,"""Pontchartrain"""
370002,600,7,2049,2.48625e7,"""Pontchartrain"""
370002,600,8,2049,2.4489e7,"""Pontchartrain"""
370003,600,7,2049,2.94156e7,"""Pontchartrain"""


In [36]:
missing_marsh_region = (
    veg_marsh_raw
    .filter(pl.col("Region").is_null())
    .select("element_id")
    .unique()
)

print(
    "Fresh-marsh elements without a region:",
    missing_marsh_region.height
)

display(missing_marsh_region.head())

Fresh-marsh elements without a region: 0


element_id
i64


### Attach project IDs and implementation periods

This section separates project and FWOA marsh records.

Project records are joined to the project–model group crosswalk (`cma.c_project_model_group`) to attach `project_id` and `implementation_period`. FWOA records are retained separately as the no-action baseline.

In [37]:
project_model_lookup = (
    c_project_model_group
    .filter(
        pl.col("model_group_id")
        .is_in(project_model_group_ids)
    )
    .select([
        "project_id",
        "model_group_id",
        "implementation_period",
    ])
    .unique()
)

veg_marsh_project = (
    veg_marsh_raw
    .filter(pl.col("model_group_id") != fwoa_model_group)
    .join(
        project_model_lookup,
        on="model_group_id",
        how="inner"
    )
)

veg_marsh_fwoa = (
    veg_marsh_raw
    .filter(pl.col("model_group_id") == fwoa_model_group)
)

print("Project marsh rows:", veg_marsh_project.shape)
print("FWOA marsh rows:", veg_marsh_fwoa.shape)

display(veg_marsh_project.head())
display(veg_marsh_fwoa.head())

Project marsh rows: (1270, 8)
FWOA marsh rows: (0, 6)


element_id,model_group_id,scenario_id,calendar_year,marsh_area_m2,Region,project_id,implementation_period
i64,i64,i64,i64,f64,str,i64,i64
370001,600,8,2049,2.36367e7,"""Pontchartrain""",2960000,2
370001,600,8,2049,2.36367e7,"""Pontchartrain""",3400000,2
370001,600,8,2049,2.36367e7,"""Pontchartrain""",370400,2
370001,600,8,2049,2.36367e7,"""Pontchartrain""",370000,2
370001,600,7,2049,2.3886e7,"""Pontchartrain""",2960000,2


element_id,model_group_id,scenario_id,calendar_year,marsh_area_m2,Region
i64,i64,i64,i64,f64,str


### Calculate annual regional fresh-marsh benefit

This section sums `marsh_area_m2` across vegetation elements for each year.

Project marsh area is summarized by:

`project_id × scenario_id × Region × implementation_period × calendar_year`

FWOA marsh area is summarized by:

`scenario_id × Region × calendar_year`

Annual fresh-marsh benefit is calculated as:

`project_annual_marsh_area_m2 − fwoa_annual_marsh_area_m2`

Rows without a matching FWOA value are identified for review.

In [38]:
project_marsh_annual = (
    veg_marsh_project
    .group_by([
        "project_id",
        "scenario_id",
        "Region",
        "implementation_period",
        "model_group_id",
        "calendar_year",
    ])
    .agg(
        pl.col("marsh_area_m2")
        .sum()
        .alias("project_annual_marsh_area_m2")
    )
)

fwoa_marsh_annual = (
    veg_marsh_fwoa
    .group_by([
        "scenario_id",
        "Region",
        "calendar_year",
    ])
    .agg(
        pl.col("marsh_area_m2")
        .sum()
        .alias("fwoa_annual_marsh_area_m2")
    )
)

freshmarsh_annual = (
    project_marsh_annual
    .join(
        fwoa_marsh_annual,
        on=[
            "scenario_id",
            "Region",
            "calendar_year",
        ],
        how="left"
    )
    .with_columns(
        (
            pl.col("project_annual_marsh_area_m2")
            - pl.col("fwoa_annual_marsh_area_m2")
        ).alias("annual_freshmarsh_benefit")
    )
)

print(freshmarsh_annual.shape)
display(freshmarsh_annual.head())

(708, 9)


project_id,scenario_id,Region,implementation_period,model_group_id,calendar_year,project_annual_marsh_area_m2,fwoa_annual_marsh_area_m2,annual_freshmarsh_benefit
i64,i64,str,i64,i64,i64,f64,f64,f64
2160000,7,"""Pontchartrain""",1,636,2025,1.97721e7,null,null
3160000,7,"""Pontchartrain""",2,677,2048,2.0484e6,null,null
2490000,8,"""Chenier Plain""",1,624,2025,4.38543e7,null,null
2580000,8,"""Chenier Plain""",2,676,2048,3.62403e7,null,null
2930200,7,"""Chenier Plain""",1,605,2025,4.06548e7,null,null


In [39]:
missing_fwoa_marsh = (
    freshmarsh_annual
    .filter(pl.col("fwoa_annual_marsh_area_m2").is_null())
    .select([
        "project_id",
        "scenario_id",
        "Region",
        "implementation_period",
        "calendar_year",
    ])
    .unique()
)

print(
    "Fresh-marsh annual rows without matching FWOA:",
    missing_fwoa_marsh.height
)

display(missing_fwoa_marsh.head())

Fresh-marsh annual rows without matching FWOA: 708


project_id,scenario_id,Region,implementation_period,calendar_year
i64,i64,str,i64,i64
900100,8,"""Barataria""",1,2026
2280000,7,"""Barataria""",1,2026
2320000,7,"""Pontchartrain""",1,2028
2670000,7,"""Barataria""",2,2047
2860100,7,"""Terrebonne""",2,2046


### Summarize fresh-marsh benefit

This section averages annual fresh-marsh benefit across modeled years at:

`project_id × ScenarioID × Region × implementation_period`

Outputs include:

- `freshmarsh_benefit`: mean annual project-minus-FWOA marsh-area benefit
- `project_freshmarsh_area_m2`: mean annual project marsh area
- `fwoa_freshmarsh_area_m2`: mean annual FWOA marsh area
- `n_freshmarsh_years`: number of modeled years included

In [40]:
freshmarsh_summary = (
    freshmarsh_annual
    .group_by([
        "project_id",
        "scenario_id",
        "Region",
        "implementation_period",
    ])
    .agg([
        pl.col("annual_freshmarsh_benefit")
        .mean()
        .alias("freshmarsh_benefit"),

        pl.col("project_annual_marsh_area_m2")
        .mean()
        .alias("project_freshmarsh_area_m2"),

        pl.col("fwoa_annual_marsh_area_m2")
        .mean()
        .alias("fwoa_freshmarsh_area_m2"),

        pl.col("calendar_year")
        .n_unique()
        .alias("n_freshmarsh_years"),
    ])
    .rename({
        "scenario_id": "ScenarioID"
    })
)

print(freshmarsh_summary.shape)
display(freshmarsh_summary)

(700, 8)


project_id,ScenarioID,Region,implementation_period,freshmarsh_benefit,project_freshmarsh_area_m2,fwoa_freshmarsh_area_m2,n_freshmarsh_years
i64,i64,str,i64,f64,f64,f64,u32
1280000,7,"""Barataria""",2,null,4.23045e7,null,1
2930100,7,"""Pontchartrain""",2,null,7.38e6,null,1
2240200,8,"""Barataria""",1,null,1.44378e7,null,1
3070000,8,"""Barataria""",2,null,8.8281e6,null,1
3410000,7,"""Barataria""",2,null,6.11307e7,null,1
…,…,…,…,…,…,…,…
2880000,8,"""Pontchartrain""",1,null,891000.0,null,1
3110000,8,"""Terrebonne""",2,null,2.69064e7,null,1
3110000,7,"""Central Coast""",2,null,3.24648e7,null,1


### Join fresh-marsh benefit with land, HSI, and cost metrics

This section joins the regional fresh-marsh summary to the combined land, HSI, and cost table using:

`project_id × ScenarioID × Region × implementation_period`

This ensures that fresh-marsh benefit is matched to the correct project, scenario, region, and implementation period.

In [41]:
final_with_freshmarsh = (
    land_plus_hsi_cost
    .join(
        freshmarsh_summary,
        on=[
            "project_id",
            "ScenarioID",
            "Region",
            "implementation_period",
        ],
        how="left"
    )
)

display(final_with_freshmarsh)

project_id,ScenarioID,Region,implementation_period,avg_annual_land_building,land_building_year50,n_years_land_building_positive,BaseID,ProjectName,TypeCode,display_id,HSI_SPSTA,HSI_OYSTE,cost_total_min,cost_total_max,freshmarsh_benefit,project_freshmarsh_area_m2,fwoa_freshmarsh_area_m2,n_freshmarsh_years
i64,i64,str,i64,f64,f64,u32,i64,str,str,str,f64,f64,f64,f64,f64,f64,f64,u32
1080000,8,"""Terrebonne""",1,9.0700e7,1.4149e10,140,1080000,"""Atchafalaya River Diversion""","""DI""","""108""",-0.011029,-0.008383,7.9435e8,7.9435e8,null,null,null,null
2860000,8,"""Terrebonne""",1,1.4748e7,7.668738e8,44,2860000,"""North Lake Mechant Marsh Creat…","""MC""","""286a""",0.001199,0.0,1.1436e9,1.1436e9,null,3.50874e7,null,1
1390100,7,"""Barataria""",1,-5.1770e6,-2.6920e8,4,1390100,"""Increase Atchafalaya Flow to T…","""DI""","""139b""",null,null,5.9343e8,5.9343e8,null,null,null,null
3330000,7,"""Barataria""",1,1.0119e6,5.26185e7,42,3330000,"""Three Ridge Restoration""","""RR""","""333""",-0.022995,-0.017725,4.9135e7,4.9135e7,null,null,null,null
3110000,7,"""Pontchartrain""",1,-2.7519e7,-1.4310e9,8,3110000,"""Black and Eloi Bay Ridge and M…","""IP""","""311""",-0.003422,0.025917,7.6719e8,7.6719e8,null,6.6249e6,null,1
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
3270000,7,"""Pontchartrain""",1,-1.7784e7,-9.2477e8,18,3270000,"""Lower Plaquemines River Sedime…","""IP""","""327""",0.002059,0.0,4.2403e9,4.2403e9,null,null,null,null
3220000,8,"""Barataria""",2,1.1484e8,1.5618e10,119,3220000,"""Freshwater Delivery to Western…","""DI""","""322""",-0.004518,-0.002076,1.4861e8,1.4861e8,null,null,null,null
2450000,8,"""Pontchartrain""",2,-3.7578e7,-1.2025e9,12,2450000,"""LaBranche Hydrologic Restorati…","""HR""","""245""",null,null,null,null,null,null,null,null


In [42]:
missing_cost_rows = final_with_freshmarsh.filter(
    pl.col("cost_total_min").is_null() | pl.col("cost_total_max").is_null()
)

display(missing_cost_rows.select([
    "project_id",
    "BaseID",
    "ScenarioID",
    "ProjectName",
    "TypeCode",
    "Region",
    "implementation_period",
]).unique().sort([
    "project_id",
    "ScenarioID",
    "implementation_period",
]))

project_id,BaseID,ScenarioID,ProjectName,TypeCode,Region,implementation_period
i64,i64,i64,str,str,str,i64
60000,60000,7,"""Lower Breton Diversion""","""DI""","""Pontchartrain""",2
60000,60000,8,"""Lower Breton Diversion""","""DI""","""Pontchartrain""",2
140000,140000,7,"""Central Wetlands Diversion""","""DI""","""Barataria""",1
140000,140000,7,"""Central Wetlands Diversion""","""DI""","""Pontchartrain""",1
140000,140000,8,"""Central Wetlands Diversion""","""DI""","""Pontchartrain""",1
…,…,…,…,…,…,…
3520000,3520000,8,"""Wildhorse Marsh Creation""","""MC""","""Chenier Plain""",2
3530000,3530000,7,"""Chenier Ridges Restoration""","""RR""","""Chenier Plain""",2
3530000,3530000,8,"""Chenier Ridges Restoration""","""RR""","""Chenier Plain""",2


In [43]:
missing_project_ids = (
    missing_cost_rows
    .get_column("project_id")
    .unique()
    .to_list()
)

missing_project_ids_sql = ",".join(map(str, missing_project_ids))

with engine.connect() as conn:
    missing_cost_check = pl.from_pandas(pd.read_sql(
        text(f"""
            SELECT
                project_id,
                scenario_id,
                implementation_period,
                cost_scenario_id,
                COUNT(*) AS n_rows,
                MIN(total_cost) AS cost_total_min,
                MAX(total_cost) AS cost_total_max
            FROM pct.vw_o_cost_project
            WHERE project_id IN ({missing_project_ids_sql})
            GROUP BY
                project_id,
                scenario_id,
                implementation_period,
                cost_scenario_id
            ORDER BY
                project_id,
                scenario_id,
                implementation_period,
                cost_scenario_id
        """),
        conn
    ))

display(missing_cost_check)

project_id,scenario_id,implementation_period,cost_scenario_id,n_rows,cost_total_min,cost_total_max
i64,i64,i64,i64,i64,f64,f64
60000,7,1,1,1,3.9900e8,3.9900e8
60000,7,1,2,1,5.1154e8,5.1154e8
60000,7,1,3,1,6.2408e8,6.2408e8
60000,8,1,1,1,3.9900e8,3.9900e8
60000,8,1,2,1,5.1154e8,5.1154e8
…,…,…,…,…,…,…
3540000,7,1,2,1,7.7396e8,7.7396e8
3540000,7,1,3,1,8.4521e8,8.4521e8
3540000,8,1,1,1,7.9622e8,7.9622e8


In [44]:
with engine.connect() as conn:
    missing_cost_periods = pl.from_pandas(pd.read_sql(
        text(f"""
            SELECT
                project_id,
                scenario_id,
                implementation_period,
                COUNT(*) AS n_rows,
                MIN(total_cost) AS min_cost,
                MAX(total_cost) AS max_cost
            FROM pct.vw_o_cost_project
            WHERE project_id IN ({missing_project_ids_sql})
            GROUP BY project_id, scenario_id, implementation_period
            ORDER BY project_id, scenario_id, implementation_period
        """),
        conn
    ))

display(missing_cost_periods)

project_id,scenario_id,implementation_period,n_rows,min_cost,max_cost
i64,i64,i64,i64,f64,f64
60000,7,1,3,3.9900e8,6.2408e8
60000,8,1,3,3.9900e8,6.2408e8
140000,7,2,3,3.1188e8,3.7553e8
140000,8,2,3,3.1188e8,3.7553e8
350000,7,2,3,1.4985e8,1.8055e8
…,…,…,…,…,…
3520000,8,1,3,9.6623e7,1.1511e8
3530000,7,1,3,4.1281e8,4.9291e8
3530000,8,1,3,4.1281e8,4.9291e8


## Write final dataset to CSV

This section creates the local `data` directory and writes `final_with_freshmarsh` to `data/joined.csv`.

In [ ]:
# final_consistent = (
#     final_with_freshmarsh
#     .filter(
#         pl.col("cost_total_min").is_not_null()
#         & pl.col("cost_total_max").is_not_null()
#     )
# )

In [ ]:
# final_consistent.select([
#     pl.len().alias("total_rows"),
#     pl.col("cost_total_min").null_count().alias("missing_cost_total_min"),
#     pl.col("cost_total_max").null_count().alias("missing_cost_total_max"),
# ])

In [45]:
from pathlib import Path

data_dir = Path.cwd() / "data"
data_dir.mkdir(parents=True, exist_ok=True)
output_path = data_dir / "joined.csv"


In [46]:
final_with_freshmarsh.write_csv(output_path)